# MSI node2vec: full local seed-0 run

Select a local **Python 3 kernel → Run All**. The first run creates an independent environment and requires network access. Later stages use prepared local inputs.

Pipeline: environment → configuration → **2,995,900 walks** → **128D embeddings, saved immediately** → **ten-fold Logistic Regression** → paper comparison.

Run one seed only. Reuse saved negatives, without parameter search or DREAMwalk training calls. This published copy has cleared outputs; historical results are summarized in the repository README.


## 1. Directories and run name

Find both model directories in the same repository. New outputs go into `01.node2vec_reproduction/`. Reuse completed walks and embeddings. Use a new `RUN_NAME` after changing inputs, source or configuration.


In [3]:
from pathlib import Path
import csv
import html
import json
import os
import shutil
import subprocess
import sys
from IPython.display import display, HTML

RUN_NAME = 'msi_seed0'
# If the server starts outside this repository, set the root or 00.Reproduction absolute path.
REPOSITORY_OVERRIDE = None

def find_reproduction_root(start):
    start = Path(start).expanduser().resolve()
    for parent in [start, *start.parents]:
        for candidate in [parent / '00.Reproduction', parent]:
            if ((candidate / '00.DREAMwalk_reproduction').is_dir()
                    and (candidate / '01.node2vec_reproduction').is_dir()):
                return candidate
    raise FileNotFoundError('00.Reproduction not found; set REPOSITORY_OVERRIDE to the repository root or 00.Reproduction.')

REPOSITORY = find_reproduction_root(REPOSITORY_OVERRIDE or Path.cwd())
ROOT = REPOSITORY / '01.node2vec_reproduction'
SOURCE = REPOSITORY / '00.DREAMwalk_reproduction'
ENV_PREFIX = ROOT / 'environment/.conda-env'
PYTHON = ENV_PREFIX / 'bin/python'
RUNNER = ROOT / 'src/run_node2vec.py'
if not RUN_NAME or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_NAME):
    raise ValueError('RUN_NAME accepts only letters, digits, underscores and hyphens')
RESULTS = ROOT / 'results' / RUN_NAME
LOGS = ROOT / 'logs' / RUN_NAME
LOGS.mkdir(parents=True, exist_ok=True)

for path in [SOURCE / 'data/processed/msi_graph.tsv',
             SOURCE / 'data/processed/msi_nodetypes.tsv',
             SOURCE / 'data/processed/msi_positive_pairs.tsv',
             SOURCE / 'results/msi_repeat0_negative_pairs.csv']:
    if not path.is_file():
        raise FileNotFoundError(f'Required input missing: {path}')
print('Project: ', ROOT)
print('Embedding：', ROOT / 'data' / RUN_NAME / 'msi_node2vec_seed0.pkl')
print('Results: ', RESULTS)


Project: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction
Embedding： /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/data/msi_seed0/msi_node2vec_seed0.pkl
Results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/results/msi_seed0


## 2. Independent environment: installed on first run

Use local mamba/conda to create a dedicated Python 3.8 environment. Installations do not alter the notebook kernel or DREAMwalk environment. Subprocess execution needs **no kernel restart or switch**. NumPy, SciPy and scikit-learn match DREAMwalk pins; add Gensim 4.3.2. Matching environments are reused. Installation logs go to `logs/`.


In [5]:
def stream_command(command, log_path, env=None):
    print('Starting: ', ' '.join(map(str, command)), flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\nCOMMAND: ' + repr(list(map(str, command))) + '\n')
        with subprocess.Popen(list(map(str, command)), cwd=ROOT, env=env,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    log.write(line)
                    log.flush()
                code = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
            if code:
                raise subprocess.CalledProcessError(code, command)

PINS = {'numpy':'1.23.3', 'scipy':'1.9.1', 'scikit-learn':'1.1.3',
        'gensim':'4.3.2', 'smart_open':'6.4.0'}
probe = (
    'import sys; from importlib.metadata import version; '
    'import numpy, scipy, sklearn, gensim; '
    'assert sys.version_info[:2] == (3,8); '
    f'expected={PINS!r}; '
    'actual={k:version(k) for k in expected}; '
    'assert actual == expected, actual; print(sys.version); print(actual)'
)
ready = subprocess.run([str(PYTHON), '-c', probe], capture_output=True, text=True) if PYTHON.exists() else None
if ready is None or ready.returncode:
    candidates = [shutil.which('mamba'), shutil.which('conda'),
                  str(Path.home()/'mambaforge/bin/mamba'),
                  str(Path.home()/'miniforge3/bin/mamba'),
                  str(Path.home()/'miniconda3/bin/conda')]
    manager = next((p for p in candidates if p and Path(p).is_file()), None)
    if manager is None:
        raise RuntimeError('mamba/conda not found; install Miniforge and rerun this section.')
    action = 'install' if (ENV_PREFIX/'conda-meta/history').exists() else 'create'
    command = [manager, action, '--yes', '--prefix', str(ENV_PREFIX),
               '--override-channels', '-c', 'conda-forge', 'python=3.8']
    command += [f'{package}={v}' for package, v in PINS.items()]
    install_env = dict(os.environ)
    install_env['CONDA_PKGS_DIRS'] = str(ROOT/'environment/.conda-pkgs')
    stream_command(command, LOGS/'environment.log', env=install_env)
    ready = subprocess.run([str(PYTHON), '-c', probe], capture_output=True, text=True)
    if ready.returncode:
        print(ready.stdout, ready.stderr)
        raise RuntimeError('Environment is not ready; inspect the installation log above.')
print(ready.stdout)


3.8.20 | packaged by conda-forge | (default, Sep 30 2024, 17:48:42) 
[Clang 17.0.6 ]
{'numpy': '1.23.3', 'scipy': '1.9.1', 'scikit-learn': '1.1.3', 'gensim': '4.3.2', 'smart_open': '6.4.0'}



## 3. Fixed configuration and method sources

| Component | Setting |
|---|---|
| Inputs | Shared MSI molecular graph; 29,959 nodes; 478,728 non-treatment rows |
| Walks | 100 per node; length 10; `p=q=1`; undirected, unit weights |
| Embeddings | 128D; window 4; four training workers; one epoch; seed 0 |
| Labels | 5,926 positives + the saved 5,926 seed-0 negatives |
| CV | Ten folds; shuffle=True; random_state=0; save pair-to-fold IDs |
| Predictor | Logistic Regression; L2; C=1; lbfgs; max_iter=1000 |
| Pair feature | Drug embedding minus disease embedding; no scaling |

Walk count/length, dimension, p/q and classifier follow Supplementary Method 1; window follows the main text. Pair subtraction follows released DREAMwalk prediction code. The baseline pair operator and complete LR settings are reconstruction choices. Standard Gensim settings are recorded explicitly; one epoch follows the node2vec reference CLI default. Gensim's dynamic window and `min_alpha=0.0001` differ from DREAMwalk C++ details.

This is a **documented node2vec reconstruction**, not the authors' recovered benchmark script. Four-thread training is not guaranteed to be bitwise deterministic. Record the inherited `PYTHONHASHSEED` without setting another startup value.


In [6]:
CHILD_ENV = dict(os.environ)
CHILD_ENV['PYTHONUNBUFFERED'] = '1'

def run_phase(phase):
    stream_command([PYTHON, '-u', RUNNER, phase, '--run-name', RUN_NAME],
                   LOGS / f'{phase}.log', env=CHILD_ENV)

run_phase('config')


Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/src/run_node2vec.py config --run-name msi_seed0
{
  "config": {
    "seed": 0,
    "p": 1.0,
    "q": 1.0,
    "num_walks": 100,
    "walk_length": 10,
    "vector_size": 128,
    "window": 4,
    "workers": 4,
    "epochs": 1,
    "sg": 1,
    "hs": 0,
    "negative": 5,
    "sample": 0.001,
    "alpha": 0.025,
    "min_alpha": 0.0001,
    "ns_exponent": 0.75,
    "min_count": 0,
    "sorted_vocab": 1,
    "batch_words": 10000,
    "shrink_windows": true,
    "graph_policy": "undirected simple unit-weight graph; all treatments excluded",
    "walk_sampler": "uniform neighbor sampling specialized to p=q=1; disk corpus",
    "pair_feature": "drug_embedding - disease_embedding",
    "classifier": {
      "penalty": "l2",
      "C": 1.0,
      "solver": "lbfgs",
      "fit_intercept": tru

## 4. Generate full walks

**Scientific computation starts here.** With `p=q=1` and unit weights, node2vec samples uniform neighbors. This specialized implementation preserves that transition distribution and streams walks to disk in one process, printing progress each round.

No node types, semantic hierarchy, teleportation or edge-type EM are used. Save numeric node tokens and their mapping.


In [7]:
run_phase('walks')


Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/src/run_node2vec.py walks --run-name msi_seed0
Walk round 1/100; 29,959 walks; 1.2s
Walk round 2/100; 59,918 walks; 1.4s
Walk round 3/100; 89,877 walks; 1.5s
Walk round 4/100; 119,836 walks; 1.7s
Walk round 5/100; 149,795 walks; 1.9s
Walk round 6/100; 179,754 walks; 2.1s
Walk round 7/100; 209,713 walks; 2.3s
Walk round 8/100; 239,672 walks; 2.4s
Walk round 9/100; 269,631 walks; 2.6s
Walk round 10/100; 299,590 walks; 2.8s
Walk round 11/100; 329,549 walks; 3.0s
Walk round 12/100; 359,508 walks; 3.2s
Walk round 13/100; 389,467 walks; 3.3s
Walk round 14/100; 419,426 walks; 3.5s
Walk round 15/100; 449,385 walks; 3.7s
Walk round 16/100; 479,344 walks; 3.9s
Walk round 17/100; 509,303 walks; 4.1s
Walk round 18/100; 539,262 walks; 4.3s
Walk round 19/100; 569,221 walks; 4.5s
Walk round 20/100; 5

## 5. Train and immediately save embeddings

Read the saved walk corpus with standard Gensim Skip-gram and four threads. Immediately save **`data/<RUN_NAME>/msi_node2vec_seed0.pkl`**. Evaluation failures do not require embedding retraining. Keep the kernel and computer awake until training finishes.


In [8]:
run_phase('embedding')
print('Saved: ', ROOT/'data'/RUN_NAME/'msi_node2vec_seed0.pkl')


Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/src/run_node2vec.py embedding --run-name msi_seed0
Reusing completed checkpoint: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/data/msi_seed0/walks.txt
2026-10-13 15:09:36,115 INFO collecting all words and their counts
2026-10-13 15:09:36,115 INFO PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-10-13 15:09:36,144 INFO PROGRESS: at sentence #10000, processed 100000 words, keeping 23454 word types
2026-10-13 15:09:36,171 INFO PROGRESS: at sentence #20000, processed 200000 words, keeping 28052 word types
2026-10-13 15:09:36,198 INFO PROGRESS: at sentence #30000, processed 300000 words, keeping 29959 word types
2026-10-13 15:09:36,223 INFO PROGRESS: at sentence #40000, processed 400000 words, keeping 29959 word types
2026-10-13 15:09:3

## 6. Ten-fold Logistic Regression

Reuse saved seed-0 negatives. Folds are **deterministically regenerated** with the existing DREAMwalk procedure and explicitly saved. Save metrics and predictions per fold, then summarize all ten. No extra seeds, pair operators or parameter search are added.


In [9]:
run_phase('evaluate')


Launching: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/environment/.conda-env/bin/python -u /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/src/run_node2vec.py evaluate --run-name msi_seed0
Reusing completed checkpoint: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/data/msi_seed0/msi_node2vec_seed0.pkl
{"seed": 0, "fold": 1, "accuracy": 0.657672849915683, "auroc": 0.7012390195905577, "aupr": 0.708569742489131, "n_train": 10666, "n_test": 1186, "converged": true, "iterations": 40, "warnings": "", "seconds": 0.05663495800000007}
{"seed": 0, "fold": 2, "accuracy": 0.636593591905565, "auroc": 0.7072450085170154, "aupr": 0.6953441805711218, "n_train": 10666, "n_test": 1186, "converged": true, "iterations": 40, "warnings": "", "seconds": 0.051547792000000925}
{"seed": 0, "fold": 3, "accuracy": 0.6531645569620254, "auroc": 0.7197683560457591, "aupr": 0.7032743313195209, "n_train": 10667, "n_test": 

## 7. Compare with the paper's node2vec baseline

Compare node2vec with **its own** Figure 2a reference, averaged from ten Source Data rows: Accuracy **0.643231**, AUROC **0.692508**, AUPR **0.691617**.

Local evaluation is one ten-fold repeat; the paper uses ten repeats. Fold SD and repeat SD measure different variation. Report LR convergence warnings without automatically adding experiments.

Agreement supports shared molecular inputs/evaluation. It does not validate MeSH/ATC mappings or establish exact unpublished baseline choices.


In [10]:
def show_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        columns = reader.fieldnames
        rows = list(reader)
    def pretty(value):
        try:
            return f'{float(value):.6f}'
        except (TypeError, ValueError):
            return str(value)
    header = ''.join(f'<th>{html.escape(c)}</th>' for c in columns)
    body = ''.join('<tr>' + ''.join(f'<td>{html.escape(pretty(row[c]))}</td>' for c in columns) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>'+header+'</tr></thead><tbody>'+body+'</tbody></table>'))

show_csv(RESULTS/'summary.csv')
show_csv(RESULTS/'paper_comparison.csv')
status = json.loads((RESULTS/'evaluation_complete.json').read_text())
print('Completed seed-0 embeddings and ten-fold evaluation.')
print('All folds converged: ', status['all_folds_converged'])
print('Results directory: ', RESULTS)
print('Embedding：', ROOT/'data'/RUN_NAME/'msi_node2vec_seed0.pkl')


metric,mean,SD,n_folds,n_repeats,all_folds_converged
accuracy,0.648245,0.016850,10.000000,1.000000,True
auroc,0.703811,0.014606,10.000000,1.000000,True
aupr,0.701662,0.014908,10.000000,1.000000,True


metric,local_seed0_10fold_mean,paper_10repeat_mean,local_minus_paper,paper_SD_between_repeats,local_SD_between_folds,all_folds_converged
accuracy,0.648245,0.643231,0.005015,0.014472,0.016850,True
auroc,0.703811,0.692508,0.011303,0.011679,0.014606,True
aupr,0.701662,0.691617,0.010045,0.017540,0.014908,True


Completed: seed 0 embedding and ten-fold evaluation.
All folds converged: True
Results directory: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/results/msi_seed0
Embedding： /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/01.node2vec_reproduction/data/msi_seed0/msi_node2vec_seed0.pkl


## Outputs and sources

- Embeddings: `data/<RUN_NAME>/msi_node2vec_seed0.pkl`.
- Results: `summary.csv`, `paper_comparison.csv`, `fold_results.csv`, `fold_assignments.csv`, `oof_predictions.csv` under `results/<RUN_NAME>/`.
- Configuration and completion markers are saved beside results; logs use `logs/<RUN_NAME>/`.
- Completed stages are reused. An interrupted CV restarts evaluation from the saved embedding.
- Use a new `RUN_NAME` after source/input/parameter changes.

[Bang et al. (2023)](https://www.nature.com/articles/s41467-023-39301-y) · [Supplement](https://media.springernature.com/original/springer-static/esm/art%3A10.1038%2Fs41467-023-39301-y/MediaObjects/41467_2023_39301_MOESM1_ESM.pdf) · [node2vec reference](https://github.com/aditya-grover/node2vec) · [Gensim Word2Vec](https://radimrehurek.com/gensim/models/word2vec.html)


## Related control: DREAMwalk no-teleport length=4

See the walk-length section of the [DREAMwalk notebook](../../00.DREAMwalk_reproduction/notebooks/MSI_DREAMwalk_local.ipynb). The node2vec run need not be repeated.
